# 21-04 · Стреляем

Практика к разделу [«Стреляем»](../../site/chapters/glava-21/21-04-strelba.html).

## Цель

Создать пулю у носа корабля и убедиться, что она улетает вверх.

## Рабочий пример

In [ ]:
import random

import pygame

SHIRINA, VYSOTA = 480, 720
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 44, 44
KORABL_SKOROST = 6

PULYA_SHIRINA, PULYA_VYSOTA = 6, 18
PULYA_SKOROST = 9

VRAG_SHIRINA, VRAG_VYSOTA = 32, 28
VRAG_SKOROST = 2
INTERVAL_POYAVLENIYA_VRAGA = 45

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    return {
        "korabl": pygame.Rect(
            SHIRINA // 2 - KORABL_SHIRINA // 2,
            VYSOTA - KORABL_VYSOTA - 20,
            KORABL_SHIRINA,
            KORABL_VYSOTA,
        ),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "kadrov_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi):
    korabl = state["korabl"]
    if klavishi[pygame.K_LEFT]:
        korabl.x -= KORABL_SKOROST
    if klavishi[pygame.K_RIGHT]:
        korabl.x += KORABL_SKOROST
    korabl.x = max(0, min(korabl.x, SHIRINA - KORABL_SHIRINA))


def vystrelit(state):
    korabl = state["korabl"]
    pulya = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append(pulya)


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA)


def obnovit_igru(state):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya.y -= PULYA_SKOROST
    state["puli"] = [p for p in state["puli"] if p.bottom > 0]

    state["kadrov_do_vraga"] -= 1
    if state["kadrov_do_vraga"] <= 0:
        state["vragi"].append(sozdat_vraga())
        state["kadrov_do_vraga"] = INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag.y += VRAG_SKOROST

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya.colliderect(vrag):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag.bottom >= VYSOTA or vrag.colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya)
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag)

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

In [ ]:
state = novaya_igra()
vystrelit(state)

print("Пуль после выстрела:", len(state["puli"]))
print("Позиция пули:", state["puli"][0])
print("Позиция корабля:", state["korabl"])

## Проверка результата

In [ ]:
pulya = state["puli"][0]
assert pulya.centerx == state["korabl"].centerx
assert pulya.top == state["korabl"].top
print("Верно: пуля появилась ровно у носа корабля.")

## Эксперимент — пуля улетает за экран и исчезает

In [ ]:
y_do = state["puli"][0].y
for kadr in range(200):
    obnovit_igru(state)

print("Пуль осталось:", len(state["puli"]))
assert len(state["puli"]) == 0, "пуля должна улететь за верхний край экрана и исчезнуть"
print(f"Верно: пуля улетела с {y_do} за пределы экрана и была удалена из списка.")